In [1]:
#Very first step is to import these modules and libraries into the satprocessing kernel
import numpy as np
import dask as ds
import earthaccess as ea
import xarray as xr
import sys
import matplotlib.pyplot as plt
import requests
import pandas as pd
import PyCO2SYS as pyco2
print("module import success")

import sys
sys.path.append(r'C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\RESOURCES\\python\\utilities\\bootstrap\\')
print("path import success")

import pprint
from environment import bootstrap_environment  
pprint.pprint(bootstrap_environment(preferred="edmund_gable"))
print("success")

from utilities.bootstrap.environment import bootstrap_environment
env = bootstrap_environment(preferred="edmund_gable")
env
print("utilities import success")

#Before this next step, install the library 'resource': conda install -c conda-forge resource
from utilities import get_dates, get_source_file_dates, find_text
find_text("stats")

from pathlib import Path

root = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026")

c:\Users\edmund.gable\.conda\envs\satprocessing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


module import success
path import success
{'dataset_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/DATASETS'),
 'hostname': 'NECL04363598',
 'lookuptable_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/lookuptables'),
 'metadata_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/metadata'),
 'project_root': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'python_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'satlogs_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/logs/satprocessing'),
 'timestamp': datetime.datetime(2026, 6, 5, 11, 42, 26, 491035),
 'utilities_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python/utilities/src/utilities'),
 'workflow_resources': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources')

Steps for opening Glider Data or Buoy data
- Download data file into github folder 'data'
- In VScode pull datset from data folder into OA_data notebook
- identify structure of file (printout variable names and dimensions (lat lon depth time))
- identify variable names, units (aragonite, pH, carbon concentration, anything else) with index
- Determine criteria of values and variables
    - Time dimension of dates but want a date range
    - ex: January 24 to February 1st
        - specify dates matching time format in datafile
        - or make a broader description if they just give a month (01-31 of month)
    - Space dimension - nail down where it occured with coordinates (lat, long) (*N *E/W) but identify specific region and if its nearshore, offshore
        - with moving data: take all data within a 'box' of coordinates (minimum and maximum of latitude and longitude is bounding box)
        - With buoy data identify specific larger region around fixed point with a centered bounding box (+- 10km for lat and lon)
    - If data does not have saturated aragonite, create function for sat aragonite from pH and CO2 ppm
        - def sat_arag(ph, CO2)
    - For fixed point: Identify mininum saturated aragonite within that date range and identify as the event value
    - For a moving point: identify average of each day within box, then find minimum of all daily averages and identify as event value
    - Identify threshold for an event and create function 
        - If sat_arag < Longfin_threshold:
            - return sat_arag
    - sat_arag>event_data
    - make replicate file with dictionary that can receive all of the sat_arag data over time and space
        - ([start date],[end date], Box:[min longitude][max longitude][min lattitude][max lattitude], [min sat_arag],[below threshold?],[approaching threshold?])

- by saving this data in a table with a known format and NAMED VARIABLES (not hardcoded), you can read in MOM6 data and compare it with a function that runs repeatedly for all events
- output a text file/dictionary with keys and values
- Fill out sheet with dictionary values


Basically same for DO but switch out sat_aragewith mg/L O2, threshold is 2.31+-0.1



In [2]:
#Step 1: Open datset and convert to a dataframe
ds=xr.open_dataset(root/ "data" / "Raw_Data" / "Gliders" / "glider_based_bottom_OA_data_2019_2025.nc")
df=ds.to_dataframe()
df
#Check the names of the columns for function inputs
print(df.columns.tolist())

['deployment', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom']


Functions List:

In [3]:

#Step 2: Define these functions
#Function to return the spatial extents for each vessel cruise or glider deployment, which will be used to create bounding boxes for the model grid
def get_extent(df, name_column, lon_column, lat_column, depth_column):
    lon_max=df.groupby(name_column)[lon_column].max()
    lon_min=df.groupby(name_column)[lon_column].min()
    lat_max=df.groupby(name_column)[lat_column].max()
    lat_min=df.groupby(name_column)[lat_column].min()
    depth_max=df.groupby(name_column)[depth_column].max()
    depth_min=df.groupby(name_column)[depth_column].min()
    extent_df=pd.DataFrame({
        "lon_max": lon_max,
        "lon_min": lon_min,
        "lat_max": lat_max,
        "lat_min": lat_min,
        "depth_max": depth_max,
        "depth_min": depth_min

    })
    return extent_df.reset_index()
#vessel_extent = get_extent(df, "deployment", "lon", "lat", 'depth')
#print(vessel_extent)
#vessel_extent.to_excel(root / "data" / "Processed_Data" / "vessel__based_bottom_OA_extent_2007_2024.xlsx", index=False)

#Function to qualify the omega aragnoite saturation states of each data point in a file
def classify_arag(df, aragonite_column):
    filtered_df=df[df[aragonite_column]<1.4].copy()
    filtered_df['aragonite_status']='suboptimal'
    filtered_df.loc[filtered_df[aragonite_column]<1.19, 'aragonite_status']='Atlantic Cod threshold surpassed'
    filtered_df.loc[filtered_df[aragonite_column]<1.1, 'aragonite_status']='Atlantic Sea Scallop threshold surpassed'
    filtered_df.loc[filtered_df[aragonite_column]<1, 'aragonite_status']='Critical threshold surpassed'
    return filtered_df

#Function to export the classified aragonite as a csv
def export_classified_arag(df, as_file_name):
    df_classified=classify_arag(df, "omega_bottom")
    df_classified
    df_classified.to_csv(root/"data"/"Processed_Data"/ as_file_name, index=False)

#Function that calculates the daily averages of saturated aragonite
def calculate_daily_averages(df, datetime_col, name_col, target_col,):
    df_working = df.copy()
    
    # 1. Extract the date
    df_working['date'] = pd.to_datetime(df_working[datetime_col]).dt.date
    
    # 2. Ensure the raw aragonite column is numeric
    df_working[target_col] = pd.to_numeric(df_working[target_col], errors='coerce')
    
    # 3. Group by vessel and date, then calculate the mean of the raw numbers
    daily_avg_df = df_working.groupby([name_col, 'date'])[target_col].mean().reset_index()
    
    # 4. Rename the column so we know it is a daily average
    avg_col_name = f'{target_col}_daily_avg'
    daily_avg_df.rename(columns={target_col: avg_col_name}, inplace=True)
    
    return daily_avg_df

#Function which will return the final data file with spatial extent of the glider/vessel deployment and averaged+classified daily values of saturated aragonite
def combine_df(df1, df2, join_key):
    combined_df=pd.merge(df1, df2, on=join_key)
    return combined_df

#Function that checks the names of columns in a dataframe
def check_columns(df):
    df.reset_index()
    print(df.columns.to_list())

In [4]:
#When performing the get_extent() function it is important to first reset the dataframe index becasue pandas often ignores time columns as an index
df=df.reset_index()
dfge=get_extent(df, 'deployment', 'lon', 'lat', 'depth')
print(dfge.columns.tolist())
df=classify_arag(df, 'omega_bottom')
print(df.columns.tolist())
df

['deployment', 'lon_max', 'lon_min', 'lat_max', 'lat_min', 'depth_max', 'depth_min']
['time', 'deployment', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom', 'aragonite_status']


,time,deployment,lat,lon,depth,bottom_depth,pH_bottom,omega_bottom,temperature_bottom,aragonite_status
0,2019-08-05 00:01:15.711111168,ru30-20190717T1812,39.809955,-73.212830,39.491982,43.0,7.8281,1.08,7.95,Atlantic Sea Scallop threshold surpassed
1,2019-08-05 00:05:55.698529024,ru30-20190717T1812,39.810494,-73.214054,39.546547,43.0,7.8257,1.08,7.95,Atlantic Sea Scallop threshold surpassed
2,2019-08-05 00:10:47.911289856,ru30-20190717T1812,39.811057,-73.215331,41.148884,43.0,7.8294,1.09,7.96,Atlantic Sea Scallop threshold surpassed
3,2019-08-05 00:15:05.200000000,ru30-20190717T1812,39.811553,-73.216456,41.044708,43.0,7.8278,1.08,7.96,Atlantic Sea Scallop threshold surpassed
4,2019-08-05 00:19:32.785124096,ru30-20190717T1812,39.812068,-73.217626,42.493248,43.0,7.8300,1.09,7.97,Atlantic Sea Scallop threshold surpassed
...,...,...,...,...,...,...,...,...,...,...
133216,2025-08-04 19:14:47.456999936,ru39-20250716T1542,38.847174,-73.664227,45.806042,48.0,7.9429,1.34,6.98,suboptimal
133217,2025-08-04 19:45:23.457999872,ru39-20250716T1542,38.849947,-73.670700,42.154625,48.0,7.9602,1.39,6.92,suboptimal
133218,2025-08-04 19:45:30.457999872,ru39-20250716T1542,38.849991,-73.670804,42.114937,48.0,7.9504,1.37,6.94,suboptimal
133554,2025-08-08 15:28:22.362999808,ru39-20250716T1542,39.109857,-74.311112,28.063700,28.0,7.8325,1.40,14.65,suboptimal


In [5]:

#dfgd is "dataframe glider daily"
dfgd=calculate_daily_averages(df, "time", "deployment", "omega_bottom")

#check the new column names to perform a classify_arag() function on the averaged dataframe
check_columns(dfgd)
#naming scheme: dfgdc = data frame glider daily classified. If vessel use dfvdc
dfgdc=classify_arag(dfgd, 'omega_bottom_daily_avg')




['deployment', 'date', 'omega_bottom_daily_avg']


In [7]:


# Step 3: Combine with your spatial bounds (min/max lat and lon)
df_final = combine_df(dfgdc, dfge, 'deployment')
print(df_final)
# Step 4: Export it 
export_path = root / "data" / "Processed_Data" / "glider_based_bottom_OA_extent_average_2019_2025.csv"
df_final.to_csv(export_path, date_format='%Y-%m-%d %H:%M:%S', index=False)

               deployment        date  omega_bottom_daily_avg  \
0     SBU01-20210513T1921  2021-05-14                1.385238   
1     SBU01-20210513T1921  2021-05-15                1.390000   
2     SBU01-20210513T1921  2021-05-23                1.372632   
3     SBU01-20210513T1921  2021-05-24                1.388421   
4     SBU01-20210513T1921  2021-05-25                1.390000   
..                    ...         ...                     ...   
538  um_242-20231109T1526  2023-12-04                1.360864   
539  um_242-20231109T1526  2023-12-05                1.355581   
540  um_242-20231109T1526  2023-12-06                1.371835   
541  um_242-20231109T1526  2023-12-07                1.380265   
542  um_242-20231109T1526  2023-12-08                1.362308   

    aragonite_status    lon_max    lon_min   lat_max   lat_min   depth_max  \
0         suboptimal -71.808568 -73.842475  40.75324  39.29104  182.633698   
1         suboptimal -71.808568 -73.842475  40.75324  39.29104 

In [8]:
def combine_avg_with_bounds(df_daily_avg, df_spatial_bounds, merge_col='vessel_name'):
    """
    Combines the averaged dataset with the maximum/minimum lat and lon dataset 
    based on the name of the glider/vessel.
    """
    # Perform a left join to attach the spatial bounds to the daily average records
    # 'how=left' ensures we keep all our daily averages even if a vessel is missing from the bounds df
    combined_df = pd.merge(df_daily_avg, df_spatial_bounds, on=merge_col, how='left')
    
    return combined_df

Classify omega_arag values and drop values >= the drop threshold (1.4).
- Returns data with lat, lon, depth, pH at each point.

In [9]:
#Full process with vessel dataset
dsv = xr.open_dataset(root / "data" / "Raw_Data" / "Vessels" / "vessel_based_bottom_OA_data_2007_2024.nc")
dfv=dsv.to_dataframe()
#print(dfv)
dfv=dfv.reset_index()
print(dfv.columns.to_list())
vessel_extent = get_extent(dfv, "cruise", "lon", "lat", 'depth')
#vessel_extent.to_excel(root / "data" / "Processed_Data" / "vessel_based_bottom_OA_extent_2007_2024.xlsx", index=False)
dfvd=calculate_daily_averages(dfv, 'time', 'cruise', 'omega_bottom')
dfvd=classify_arag(dfvd, 'omega_bottom_daily_avg')
dfv_final=combine_df(dfvd, vessel_extent, 'cruise')
print(dfv_final)
# Step 4: Export it 
export_path = root / "data" / "Processed_Data" / "vessel_based_bottom_OA_extent_average_2007_2024.csv"
dfv_final.to_csv(export_path, date_format='%Y-%m-%d %H:%M:%S', index=False)


['time', 'data_source', 'cruise', 'obs_type', 'accession', 'lat', 'lon', 'depth', 'bottom_depth', 'pH_bottom', 'omega_bottom', 'temperature_bottom']
     cruise        date  omega_bottom_daily_avg  \
0    DE1202  2012-02-08                1.136667   
1    DE1202  2012-02-09                1.300000   
2    DE1202  2012-02-10                1.395000   
3    DE1202  2012-02-11                1.236667   
4     ECOA1  2015-06-21                1.386667   
..      ...         ...                     ...   
109  PC2305  2023-10-28                1.310000   
110  PC2305  2023-11-09                1.110000   
111  PC2305  2023-11-10                1.200000   
112  PC2305  2023-11-11                1.070000   
113  PC2305  2023-11-13                1.380000   

                             aragonite_status  lon_max  lon_min  lat_max  \
0            Atlantic Cod threshold surpassed -65.7733 -75.5333  43.8550   
1                                  suboptimal -65.7733 -75.5333  43.8550   
2         

In [ ]:

ds=xr.open_dataset(root/ "data" / "Raw_Data" / "Gliders" / "glider_based_bottom_OA_data_2019_2025.nc")
df=ds.to_dataframe()
glider_extent=get_extent(df,"deployment","lon","lat", 'bottom_depth')
#glider_extent.to_excel(root/"data"/"Processed_Data"/"glider_based_bottom_OA_extent_2019_2025.xlsx")

Discontinued Code:

In [ ]:
#def classify_omega_arag(omega_arag, var_name="omega_arag", drop_above=1.4,
                        status_name="omega_arag_status"):
    import pandas as pd

    # Store original dataset for coordinate extraction
    original_ds = omega_arag if isinstance(omega_arag, xr.Dataset) else None

    if isinstance(omega_arag, xr.Dataset):
        if var_name in omega_arag:
            omega_arag = omega_arag[var_name]
        elif "omega_bottom" in omega_arag:
            omega_arag = omega_arag["omega_bottom"]
        else:
            raise ValueError(f"Dataset does not contain '{var_name}' or 'omega_bottom'")

    if isinstance(omega_arag, pd.DataFrame):
        if var_name not in omega_arag:
            raise ValueError(f"DataFrame does not contain column '{var_name}'")
        selected = omega_arag.loc[omega_arag[var_name] < drop_above].copy()
        selected[status_name] = pd.cut(
            selected[var_name],
            bins=[-float('inf'), 1.0, 1.1, 1.19, drop_above],
            labels=[
                "critical threshold surpassed",
                "Atlantic Sea Scallop threshold surpassed",
                "Atlantic Cod threshold surpassed",
                "suboptimal",
            ],
            right=False,
        ).astype(str)
        return selected

    # Apply filter
    selected = omega_arag.where(omega_arag < drop_above, drop=True)

    status = xr.where(
        selected < 1.0,
        "critical threshold surpassed",
        xr.where(
            selected < 1.1,
            "Atlantic Sea Scallop threshold surpassed",
            xr.where(
                selected < 1.19,
                "Atlantic Cod threshold surpassed",
                "suboptimal",
            ),
        ),
    )

    # Create output dataset with variables
    output_vars = {var_name: selected, status_name: status}
    
    # Add any available coordinates from the original dataset
    output_coords = {}
    if original_ds is not None:
        for coord_name in original_ds.coords:
            if coord_name not in ['time']:  # Skip time if it's a dimension
                try:
                    output_coords[coord_name] = original_ds.coords[coord_name]
                except:
                    pass
        
        # Also add any other data variables that might be useful (pH, etc.)
        for var in original_ds.data_vars:
            if var not in [var_name, status_name]:
                try:
                    # Filter the variable using the same mask
                    filtered_var = original_ds[var].where(omega_arag < drop_above, drop=True)
                    output_vars[var] = filtered_var
                except:
                    pass

    return xr.Dataset(output_vars, coords=output_coords)

Convert classified xarray Dataset to pandas DataFrame with coordinates as columns.

Average omega_arag by day and classify each day based on the average value.
- Returns a DataFrame with date, average omega_arag, and status.

In [13]:
#def average_OA_by_day(omega_arag, var_name="omega_arag", drop_above=1.4):
    import pandas as pd
    import xarray as xr

    if isinstance(omega_arag, xr.Dataset):
        if var_name in omega_arag:
            omega_arag = omega_arag[var_name]
        elif "omega_bottom" in omega_arag:
            omega_arag = omega_arag["omega_bottom"]
        else:
            raise ValueError(f"Dataset does not contain '{var_name}' or 'omega_bottom'")

    if isinstance(omega_arag, xr.DataArray):
        df = omega_arag.to_dataframe(name=var_name).reset_index()
    elif isinstance(omega_arag, pd.DataFrame):
        df = omega_arag.copy()
        if var_name not in df.columns:
            if df.shape[1] == 1:
                df = df.rename(columns={df.columns[0]: var_name})
            else:
                raise ValueError(f"DataFrame does not contain column '{var_name}'")
    else:
        raise ValueError("omega_arag must be an xarray Dataset/DataArray or a pandas DataFrame")

    if 'time' not in df.columns:
        raise ValueError("Input data must contain a 'time' column or coordinate for averaging by day")

    df['time'] = pd.to_datetime(df['time'])

    def find_column(candidates):
        for col in candidates:
            if col in df.columns:
                return col
        return None

    latitude_col = find_column(['latitude', 'lat', 'Latitude', 'LATITUDE'])
    longitude_col = find_column(['longitude','lat', 'lon', 'Longitude', 'LON'])
    ph_col = find_column(['pH', 'ph', 'PH'])
    depth_col = find_column(['depth', 'Depth', 'DEPTH'])
    cruise_col = find_column(['cruise', 'Cruise', 'CRUISE'])
    datasource_col = find_column(['datasource', 'source', 'data_source', 'data source'])
    accession_col = find_column(['accession', 'Accession', 'ACCESSION'])
    obs_type_col = find_column(['obs_type', 'obstype', 'obs type', 'OBS_TYPE'])

    agg_dict = {var_name: 'mean'}
    if latitude_col is not None:
        agg_dict[latitude_col] = ['min', 'max']
    if longitude_col is not None:
        agg_dict[longitude_col] = ['min', 'max']
    if ph_col is not None:
        agg_dict[ph_col] = 'mean'
    if depth_col is not None:
        agg_dict[depth_col] = 'mean'
    if cruise_col is not None:
        agg_dict[cruise_col] = 'first'
    if datasource_col is not None:
        agg_dict[datasource_col] = 'first'
    if accession_col is not None:
        agg_dict[accession_col] = 'first'
    if obs_type_col is not None:
        agg_dict[obs_type_col] = 'first'

    daily_avg = df.groupby(df['time'].dt.date).agg(agg_dict).reset_index()
    daily_avg.rename(columns={'time': 'date'}, inplace=True)

    if isinstance(daily_avg.columns, pd.MultiIndex):
        daily_avg.columns = ['_'.join(col).strip('_') for col in daily_avg.columns.values]

    rename_map = {
        var_name: 'average_omega',
        ph_col: 'average_pH',
        depth_col: 'average_depth',
        f'{latitude_col}_min' if latitude_col else None: 'min_latitude',
        f'{latitude_col}_max' if latitude_col else None: 'max_latitude',
        f'{longitude_col}_min' if longitude_col else None: 'min_longitude',
        f'{longitude_col}_max' if longitude_col else None: 'max_longitude',
        cruise_col: 'cruise',
        datasource_col: 'datasource',
        accession_col: 'accession',
        obs_type_col: 'obs_type',
    }
    rename_map = {k: v for k, v in rename_map.items() if k is not None}
    daily_avg = daily_avg.rename(columns=rename_map)

    daily_avg['status'] = pd.cut(
        daily_avg['average_omega'],
        bins=[-float('inf'), 1.0, 1.1, 1.19, drop_above],
        labels=[
            "critical threshold surpassed",
            "Atlantic Sea Scallop threshold surpassed",
            "Atlantic Cod threshold surpassed",
            "suboptimal",
        ],
        right=False,
    ).astype(str)

    daily_avg = daily_avg.dropna(subset=['status'], inplace=True)
    daily_avg = daily_avg[daily_avg['status'] != 'nan']

    return daily_avg

IndentationError: unexpected indent (637255342.py, line 2)